# PolyRoute — Kaggle Execution Notebook

**Multilingual Speech Routing via Dual-Signal Language ID**  
GPU: T4 x2  |  Run cells top-to-bottom in order.

| Step | What it does | Est. time |
|---|---|---|
| 0 | Env setup, datasets version fix | <1 min |
| 1 | Download & cache all models (~10 GB) | ~15 min |
| 2 | Pipeline smoke test (silence → chain) | ~3 min |
| 3 | LID evaluation (30 langs × 50 samples) | ~20 min |
| 4 | Full ASR evaluation (30 langs × 30 samples) | ~40 min |
| 5 | Baselines B1-B4 | ~60 min |
| 6 | Learned routing policy | ~30 min |
| 6b | CER-oracle routing policy (~5 min, reuses Step 4) | ~5 min |
| 7 | Ablations A1-A9 | ~135 min |


## Step 0 — Environment Setup

Run this cell first. If it prints **'Restart required'**, go to **Kernel → Restart & Run All**.

In [ ]:
# ── Step 0: Environment Setup ────────────────────────────────────────────
import sys, os, pathlib, subprocess, importlib.metadata

# Find the repo root (directory that contains src/)
def _find_repo():
    # Check explicit known paths first
    candidates = [
        '/kaggle/working/LID-Router',
        '/kaggle/input/polyroute',
        '/kaggle/input/end-sem-project',
    ]
    for d in candidates:
        if os.path.isdir(os.path.join(d, 'src')):
            return d
    # Walk both /kaggle/working and /kaggle/input subdirectories
    for base in ['/kaggle/working', '/kaggle/input']:
        if os.path.isdir(base):
            for name in sorted(os.listdir(base)):
                p = os.path.join(base, name)
                if os.path.isdir(os.path.join(p, 'src')):
                    return p
    raise RuntimeError('Cannot find repo root. Make sure your dataset is attached and contains src/.')

REPO = _find_repo()
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)
print('Repo root:', REPO)
print('Working dir:', os.getcwd())

# datasets version check (google/fleurs needs < 4.0)
ver = importlib.metadata.version('datasets')
major = int(ver.split('.')[0])
print('datasets version:', ver)
if major >= 4:
    print('Downgrading datasets to 2.20.0 (required for google/fleurs)...')
    subprocess.run(['pip', 'install', 'datasets==2.20.0',
                    '--quiet', '--force-reinstall'], check=True)
    print('**** Restart required. Go to Kernel -> Restart & Run All ****')
else:
    print('OK: datasets version compatible')

# Install any missing packages
subprocess.run(['pip', 'install', 'openai-whisper', 'speechbrain',
                'jiwer', '--quiet'], check=True)
print('OK: extra packages installed')


## Step 1 — Download & Cache Models (~15 min, ~10 GB)

Run once per Kaggle session. Models are cached in `/root/.cache/huggingface/`.

In [ ]:
# ── Step 1: Download & cache all models ──────────────────────────────────
from transformers import Wav2Vec2ForSequenceClassification, AutoFeatureExtractor
from transformers import Wav2Vec2ForCTC, AutoProcessor
import whisper

# MMS-LID-4017  (~4 GB)
print('Downloading MMS-LID-4017...')
AutoFeatureExtractor.from_pretrained('facebook/mms-lid-4017')
Wav2Vec2ForSequenceClassification.from_pretrained('facebook/mms-lid-4017')
print('OK: MMS-LID-4017')

# MMS-1b-all ASR  (~4 GB)
print('Downloading MMS-1b-all...')
AutoProcessor.from_pretrained('facebook/mms-1b-all')
Wav2Vec2ForCTC.from_pretrained('facebook/mms-1b-all')
print('OK: MMS-1b-all')

# Whisper large-v3  (~3 GB)
print('Downloading Whisper large-v3...')
whisper.load_model('large-v3')
print('OK: Whisper large-v3')

print('All models cached.')


## Step 2 — Pipeline Smoke Test

Loads all models and runs a silent audio clip through the full chain. Should complete in ~3 min.

In [ ]:
# ── Step 2: Pipeline smoke test ──────────────────────────────────────────
import numpy as np
from src.pipeline import Pipeline

pipe = Pipeline()
pipe.load_models()  # ~2-3 min on T4

# Silence -> just checks the chain doesn't crash
dummy = np.zeros(16000, dtype=np.float32)
out = pipe.run(dummy)
print('language=%s, mode=%s, backend=%s' % (out.detected_language, out.routing_mode, out.backend_used))

pipe.unload_models()
print('OK: smoke test passed')


## Step 3 — LID Evaluation (30 langs × 50 samples ≈ 1500 utterances, ~20 min)

In [ ]:
# ── Step 3: LID-only evaluation ──────────────────────────────────────────
from src.pipeline import Pipeline
from evaluation.evaluate import evaluate_lid_only
from evaluation.data_loader import SUBSET_30_FLEURS
import json, pathlib

pipe = Pipeline()
pipe.load_models()

results = evaluate_lid_only(
    pipeline=pipe,
    lang_codes=SUBSET_30_FLEURS,
    split='validation',
    max_per_lang=50,
)
print('LID Accuracy: %.3f' % results.lid_accuracy())
print('Top-3 Accuracy: %.3f' % results.lid_top3_accuracy())

pathlib.Path('results').mkdir(exist_ok=True)
with open('results/lid_only_results.json', 'w') as f:
    json.dump(results.to_dict(), f, indent=2)
print('Saved results/lid_only_results.json')

pipe.unload_models()


## Step 4 — Full ASR Evaluation (30 langs × 30 samples, ~40 min)

In [ ]:
# ── Step 4: Full pipeline evaluation ─────────────────────────────────────
from evaluation.evaluate import evaluate_full
from evaluation.data_loader import SUBSET_30_FLEURS

pipe = Pipeline()
pipe.load_models()

results = evaluate_full(
    pipeline=pipe,
    lang_codes=SUBSET_30_FLEURS,
    split='test',
    max_per_lang=30,
    save_path='results/eval_results.json',
)
print('CER: %.3f' % results.mean_cer())
print('WER: %.3f' % results.mean_wer())
print('Routing distribution:', results.routing_distribution())

pipe.unload_models()


## Step 5 — Baselines B1-B4 (~60 min total)

In [ ]:
# ── Step 5: Baselines ────────────────────────────────────────────────────
from evaluation.evaluate import (
    evaluate_baseline_oracle,
    evaluate_baseline_whisper_auto,
    evaluate_baseline_static_mms,
    evaluate_baseline_static_sb_whisper,
)
from evaluation.data_loader import SUBSET_30_FLEURS
import json

# B1: Oracle (upper bound)
r_b1 = evaluate_baseline_oracle(lang_codes=SUBSET_30_FLEURS, max_per_lang=30)
# B2: Whisper auto-detect (no LID)
r_b2 = evaluate_baseline_whisper_auto(lang_codes=SUBSET_30_FLEURS, max_per_lang=30)
# B3: Always MMS-1b-all
r_b3 = evaluate_baseline_static_mms(lang_codes=SUBSET_30_FLEURS, max_per_lang=30)
# B4: SpeechBrain LID + Whisper
r_b4 = evaluate_baseline_static_sb_whisper(lang_codes=SUBSET_30_FLEURS, max_per_lang=30)

# ── Step 5 FIX: Save baseline results (already computed above) ──

for name, r in [('B1_oracle', r_b1), ('B2_whisper_auto', r_b2),
                ('B3_static_mms', r_b3), ('B4_sb_whisper', r_b4)]:
    with open('results/%s.json' % name, 'w') as f:
        json.dump(r, f, indent=2)                    # r is already a dict
    overall_cer = r.get('overall', float('nan'))
    print('%s: CER=%.3f' % (name, overall_cer))


## Step 6 — Learned Routing Policy (~30 min)

In [ ]:
# ── Step 6: Learned routing policy ───────────────────────────────────────
from src.routing.policy_learned import generate_oracle_labels, LearnedRoutingPolicy
from evaluation.data_loader import load_fleurs, iterate_fleurs, SUBSET_30_FLEURS
from src.pipeline import Pipeline
import numpy as np, pathlib

pipe = Pipeline()
pipe.load_models()

fused_probs_list, uncertainty_list, true_langs = [], [], []
datasets = load_fleurs(SUBSET_30_FLEURS, split='validation', streaming=True)
for audio, sr, fleurs_code, true_lang, ref_text in iterate_fleurs(datasets, max_per_lang=50):
    lid_out = pipe.run_lid_only(audio, sr)
    fused_probs_list.append(lid_out['fused_probs'])
    uncertainty_list.append(lid_out['uncertainty'])
    true_langs.append(true_lang)

pipe.unload_models()

X, y = generate_oracle_labels(fused_probs_list, uncertainty_list, true_langs)
print('Training data: %s, label distribution: %s' % (X.shape, str(np.bincount(y))))

policy = LearnedRoutingPolicy()
# Improved architecture: input_dim auto-inferred from X.shape (11), BatchNorm + cosine LR, 100 epochs
history = policy.train_policy(X, y, epochs=100, lr=0.001)
print('Final val_acc: %.3f' % history['val_acc'][-1])

pathlib.Path('models').mkdir(exist_ok=True)
policy.save('models/routing_policy.pt')
print('Saved models/routing_policy.pt')

## Step 6b — CER-Oracle Routing Policy (~5 min)

Trains a second policy where routing labels come from **actual CER** on the test set
rather than LID accuracy. Reuses Step 4 output — no re-run needed.

Labels: CER < 0.15 → Mode A, CER < 0.35 → Mode B, CER ≥ 0.35 → Mode C

In [ ]:
# ── Step 6b: CER-oracle routing policy (reuses Step 4 results) ──────────
import json, pathlib
import numpy as np
from src.routing.policy_learned import LearnedRoutingPolicy

with open('results/eval_results.json') as f:
    data = json.load(f)

records = data.get('records', [])
if not records:
    raise RuntimeError('No per-utterance records in eval_results.json — '
                       'make sure Step 4 ran with the updated code.')

X_cer, y_cer = [], []
for rec in records:
    uvec = rec.get('uncertainty_vec')
    if uvec is None:
        continue
    cer = rec['cer']
    # CER-oracle labels: low CER → Mode A, mid → Mode B, high → Mode C
    label = 0 if cer < 0.15 else (1 if cer < 0.35 else 2)
    X_cer.append(uvec)
    y_cer.append(label)

X_cer = np.array(X_cer)
y_cer = np.array(y_cer)
print('CER-oracle training data: %s, label distribution: %s' % (X_cer.shape, str(np.bincount(y_cer))))

policy_cer = LearnedRoutingPolicy()
history_cer = policy_cer.train_policy(X_cer, y_cer, epochs=50, lr=0.001)
print('Final val_acc: %.3f' % history_cer['val_acc'][-1])

pathlib.Path('models').mkdir(exist_ok=True)
policy_cer.save('models/routing_policy_cer.pt')
print('Saved models/routing_policy_cer.pt')

## Step 7 — Ablations A1-A9 (~135 min)

A1–A8: component ablations. A9: CER-oracle policy comparison (requires Step 6b to complete first).
A6 rule-based half reuses Step 4 output (~40 min saved).

**Resume safe:** re-running this cell skips any ablations whose output JSON already exists in `results/ablations/`.

In [ ]:
# ── Step 7: Ablation study A1-A9 (with resume support) ──────────────────
import pathlib, json
from evaluation.ablation import run_all_ablations

SAVE_DIR = 'results/ablations'

# Check which ablations are already done
_output_files = {
    'A1': 'a1_mms_lid_only.json',
    'A2': 'a2_whisper_lid_only.json',
    'A3': 'a3_ecapa_lid.json',
    'A4': 'a4_no_routing.json',
    'A5': 'a5_no_confusion.json',
    'A6': 'a6_learned_policy.json',
    'A7': 'a7_mms_only_asr.json',
    'A8': 'a8_whisper_only_asr.json',
    'A9': 'a9_cer_oracle_policy.json',
}

done = [k for k, v in _output_files.items()
        if (pathlib.Path(SAVE_DIR) / v).exists()]
remaining = [k for k in _output_files if k not in done]

print('Already completed:', done if done else 'None')
print('Will run:', remaining if remaining else 'None — all done!')

if remaining:
    # run_all_ablations auto-skips completed runs via file checks
    run_all_ablations(max_per_lang=30, save_dir=SAVE_DIR)
    print('Ablation study complete. Results in', SAVE_DIR)
else:
    print('All ablations already complete. Results in', SAVE_DIR)

## Step 8 — Track 3: Confidence-Weighted MMS Fallback (~80 min)

**What changed (zero-compute logic fix):**
1. `backend_selector.py` now uses a per-language quality table (`config/backend_quality.yaml`) instead of always picking Whisper for Mode A.
2. Only 6/28 languages genuinely prefer Whisper (arb, cmn, hin, jpn, srp, urd). The other 22 use MMS.
3. When LID confidence < 0.5, backend defaults to MMS (degrades more gracefully on wrong-language input).
4. Mode B multi-hypothesis now tries **both** backends for the top-1 candidate and keeps the better one.

**Expected impact:** CER should drop significantly (target: beat B3's 0.132).

In [ ]:
# ── Step 8 Setup: Reload Track 3 modules ────────────────────────────────
# Run this cell once before running Step 8a or Step 8b (or both).

import json, importlib
from evaluation.evaluate import evaluate_full
from evaluation.data_loader import SUBSET_30_FLEURS

import src.asr.backend_selector as _bs
import src.decoding.single_decode as _sd
import src.decoding.multi_hypothesis as _mh
importlib.reload(_bs)
importlib.reload(_sd)
importlib.reload(_mh)
_bs._backend_prefs = None   # reset singleton so YAML is re-read

print('Track 3 modules reloaded.')


## Step 8a — Rules Policy + Track 3 (~40 min)

Re-evaluates the **rules-based routing policy** with the Track 3 confidence-weighted backend selector.  
Saves to `results/step8_track3_rules.json`.

In [ ]:
# ── Step 8a: Rules-based policy with Track 3 backend selector ──────────
from src.pipeline import Pipeline

pipe_rules = Pipeline(routing_policy='rules')
pipe_rules.load_models()

results_rules = evaluate_full(
    pipeline=pipe_rules,
    lang_codes=SUBSET_30_FLEURS,
    split='test',
    max_per_lang=30,
    save_path='results/step8_track3_rules.json',
)
print('\n=== Step 8a: Track 3 + Rules Policy ===')
print('CER: %.3f' % results_rules.mean_cer())
print('WER: %.3f' % results_rules.mean_wer())
print('Routing:', results_rules.routing_distribution())

pipe_rules.unload_models()


## Step 8b — Learned Policy + Track 3 (~40 min)

Re-evaluates the **learned MLP routing policy** (from Step 6) with the Track 3 confidence-weighted backend selector.  
> **Requires:** `models/routing_policy.pt` (trained in Step 6).  

Saves to `results/step8_track3_learned.json`.

In [ ]:
# ── Step 8b: Learned MLP policy with Track 3 backend selector ──────────
from src.pipeline import Pipeline

pipe_learned = Pipeline(routing_policy='learned')
pipe_learned.routing_agent.load_learned_policy('models/routing_policy.pt')
pipe_learned.load_models()

results_learned = evaluate_full(
    pipeline=pipe_learned,
    lang_codes=SUBSET_30_FLEURS,
    split='test',
    max_per_lang=30,
    save_path='results/step8_track3_learned.json',
)
print('\n=== Step 8b: Track 3 + Learned Policy ===')
print('CER: %.3f' % results_learned.mean_cer())
print('WER: %.3f' % results_learned.mean_wer())
print('Routing:', results_learned.routing_distribution())

pipe_learned.unload_models()


## Step 8 — Comparison Table

Run after both 8a and 8b have completed (reads from saved JSON files).

In [ ]:
# ── Step 8 Comparison: Track 3 vs previous baselines ────────────────────
import json

def _load_result(path):
    try:
        with open(path) as f: d = json.load(f)
        return d.get('overall_mean_cer', d.get('overall', float('nan'))), \
               d.get('overall_mean_wer', float('nan'))
    except Exception:
        return float('nan'), float('nan')

print('=' * 65)
print('COMPARISON: Track 3 vs Previous Results')
print('=' * 65)
print(f'{"System":<35} {"CER":>8} {"WER":>8}')
print('-' * 65)
for name, path in [
    ('B3 Static MMS',       'results/B3_static_mms.json'),
    ('A6 Learned (old)',    'results/ablations/a6_learned_policy.json'),
    ('Step4 Rules (old)',   'results/eval_results.json'),
]:
    cer, wer = _load_result(path)
    print(f'{name:<35} {cer:>8.3f} {wer:>8.3f}')
print('-' * 65)
cer_r, wer_r = _load_result('results/step8_track3_rules.json')
cer_l, wer_l = _load_result('results/step8_track3_learned.json')
print(f'{"Track 3 + Rules (NEW)":<35} {cer_r:>8.3f} {wer_r:>8.3f}')
print(f'{"Track 3 + Learned (NEW)":<35} {cer_l:>8.3f} {wer_l:>8.3f}')
print('=' * 65)
try:
    with open('results/step8_track3_learned.json') as f: d = json.load(f)
    per_lang = d.get('per_language', {})
    if per_lang:
        print('\nPer-language CER (Track 3 + Learned):')
        for lang in sorted(per_lang):
            print(f'  {lang}: CER={per_lang[lang]["mean_cer"]:.3f}')
except Exception as e:
    print(f'(Per-language breakdown unavailable: {e})')


---
## Step 9 — Phase 1+2: Script-Aware Routing + Temperature Scaling (~80 min total)

**Changes already made to src/ (no extra downloads needed):**

| File | What changed |
|------|-------------|
| `config/confusion_clusters.yaml` | `force_mode_b_threshold` per cluster: urd=0.97, srp=0.92 |
| `src/routing/confusion_map.py` | Exposes `force_mode_b_threshold`, `temperature`, `rerank_by_script`, `expected_script` |
| `src/routing/policy_rules.py` | Phase 2 temperature scaling + Phase 1 forced-Mode-B override |
| `src/routing/policy_learned.py` | Same post-decision override after MLP prediction |
| `src/routing/agent.py` | Shares `ConfusionMap` between both policies |
| `src/decoding/multi_hypothesis.py` | Script-aware reranking: Perso-Arabic wins for urd, Cyrillic wins for srp |

**Run: 9-Setup → 9a → 9b (in order)**

In [ ]:
# ── Step 9 Setup: Reload ALL Phase 1+2 modules (run before 9a and 9b) ───────
import importlib, sys

_to_reload = [
    'src.routing.confusion_map',
    'src.routing.policy_rules',
    'src.routing.policy_learned',
    'src.routing.agent',
    'src.asr.backend_selector',
    'src.decoding.multi_hypothesis',
    'src.decoding.single_decode',
    'src.pipeline',
]
for mod in _to_reload:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

import src.asr.backend_selector as _bs
_bs._backend_prefs = None   # reset cache so updated YAML is re-read

from src.routing.confusion_map import ConfusionMap
cmap = ConfusionMap()
print('Phase 1+2 modules reloaded.')
print(f'  urd  force_mode_b_threshold = {cmap.force_mode_b_threshold("urd")}')
print(f'  srp  force_mode_b_threshold = {cmap.force_mode_b_threshold("srp")}')
print(f'  urd  rerank_by_script       = {cmap.rerank_by_script("urd")}')
print(f'  urd  expected_script        = {cmap.expected_script("urd")}')
print(f'  srp  expected_script        = {cmap.expected_script("srp")}')
print(f'  ind  force_mode_b_threshold = {cmap.force_mode_b_threshold("ind")}')
print(f'  eng  force_mode_b_threshold = {cmap.force_mode_b_threshold("eng")}')
print('Sanity check passed — ready.')


## Step 9a — Rules Policy + Phase 1+2 (~40 min)

In [ ]:
# ── Step 9a: Rules policy + Phase 1+2 ───────────────────────────────────────
from src.pipeline import Pipeline
from evaluation.evaluate import evaluate_full
from evaluation.data_loader import SUBSET_30_FLEURS

pipe_9a = Pipeline(routing_policy='rules')
pipe_9a.load_models()

results_9a = evaluate_full(
    pipeline=pipe_9a,
    lang_codes=SUBSET_30_FLEURS,
    split='test',
    max_per_lang=30,
    save_path='results/step9_phase12_rules.json',
)
print('\n=== Step 9a: Phase 1+2 + Rules Policy ===')
print('CER: %.4f' % results_9a.mean_cer())
print('WER: %.4f' % results_9a.mean_wer())
print('Routing:', results_9a.routing_distribution())

pipe_9a.unload_models()

import json
with open('results/step9_phase12_rules.json') as f:
    d = json.load(f)
per = d.get('per_language', {})
print('\nProblem languages:')
for lang in ['urd', 'srp', 'hin', 'cmn', 'lao', 'jpn', 'yor']:
    if lang in per:
        print(f'  {lang}: CER={per[lang]["mean_cer"]:.4f}  LID={per[lang]["lid_accuracy"]:.2f}')


## Step 9b — Learned Policy + Phase 1+2 (~40 min)

> **Requires:** `models/routing_policy.pt` from Step 6.
> No retraining — the Phase 1+2 overrides are pure Python post-decision logic.

In [ ]:
# ── Step 9b: Learned policy + Phase 1+2 ─────────────────────────────────────
from src.pipeline import Pipeline
from evaluation.evaluate import evaluate_full
from evaluation.data_loader import SUBSET_30_FLEURS

pipe_9b = Pipeline(routing_policy='learned')
pipe_9b.routing_agent.load_learned_policy('models/routing_policy.pt')
pipe_9b.load_models()

results_9b = evaluate_full(
    pipeline=pipe_9b,
    lang_codes=SUBSET_30_FLEURS,
    split='test',
    max_per_lang=30,
    save_path='results/step9_phase12_learned.json',
)
print('\n=== Step 9b: Phase 1+2 + Learned Policy ===')
print('CER: %.4f' % results_9b.mean_cer())
print('WER: %.4f' % results_9b.mean_wer())
print('Routing:', results_9b.routing_distribution())

pipe_9b.unload_models()

import json
with open('results/step9_phase12_learned.json') as f:
    d = json.load(f)
per = d.get('per_language', {})
print('\nProblem languages after Phase 1+2:')
for lang in ['urd', 'srp', 'hin', 'cmn', 'lao', 'jpn', 'yor', 'amh']:
    if lang in per:
        print(f'  {lang}: CER={per[lang]["mean_cer"]:.4f}  LID={per[lang]["lid_accuracy"]:.2f}')


## Step 9 — Comparison vs Step 8

In [ ]:
# ── Step 9 Comparison ────────────────────────────────────────────────────────
import json

def _cer_wer(path):
    try:
        with open(path) as f: d = json.load(f)
        return (d.get('overall_mean_cer', d.get('overall', float('nan'))),
                d.get('overall_mean_wer', float('nan')))
    except Exception:
        return float('nan'), float('nan')

b3_cer, _ = _cer_wer('results/B3_static_mms.json')

print('=' * 70)
print(f'  {"System":<40} {"CER":>8} {"WER":>8}  Beat B3?')
print('-' * 70)
for name, path in [
    ('B3 Static MMS (target)',        'results/B3_static_mms.json'),
    ('A6 Learned (old baseline)',     'results/ablations/a6_learned_policy.json'),
    ('Step8 Track3 Learned',          'results/step8_track3_learned.json'),
    ('Step9a Phase1+2 Rules',         'results/step9_phase12_rules.json'),
    ('Step9b Phase1+2 Learned ★',    'results/step9_phase12_learned.json'),
]:
    cer, wer = _cer_wer(path)
    ok = '✅' if cer < b3_cer else '❌' if cer == cer else '⏳'
    print(f'  {name:<40} {cer:>8.4f} {wer:>8.4f}  {ok}')
print('=' * 70)


---
## Step 10 — Phase 3: F₀ Pitch Features + MLP Retrain (~60 min)

**What this step does:**
1. Installs `pyworld` (WORLD vocoder — CPU only, ~1 min install)
2. Re-collects dev-set training data augmented with 4-dim F₀ features:
   `[mean_F₀, F₀_std, F₀_range, voiced_ratio]`
3. Retrains the routing MLP with `input_dim=15` (11 base + 4 F₀)
4. Evaluates the F₀-aware pipeline on the test set

**Why F₀ helps distinguish routing for tonal languages:**
- `cmn` (4 tones): high F₀_std → route confidently to MMS
- `lao` (6 tones): very high F₀_std → MMS preferred
- `jpn` (pitch-accent): low F₀_std → distinct from cmn/lao
- `yor` (tonal): elevated F₀_std vs European langs

> Run: 10-Setup → 10a → 10b → 10c → 10-Comparison

In [ ]:
# ── Step 10 Setup: Install pyworld ──────────────────────────────────────────
import subprocess, sys, importlib

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'pyworld', '--quiet'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('pyworld installed successfully.')
else:
    print('pyworld install warning:', result.stderr[:400])

# Reload f0_features now that pyworld may be available
if 'src.lid.f0_features' in sys.modules:
    importlib.reload(sys.modules['src.lid.f0_features'])

from src.lid.f0_features import extract_f0_features, is_available
print(f'F0 extraction active: {is_available()}')

if is_available():
    import numpy as np
    test_audio = np.random.randn(16000).astype(np.float32) * 0.01
    feats = extract_f0_features(test_audio, 16000)
    print(f'Test feature vector: {feats}  (shape={feats.shape})')
    print('Fields: [mean_f0, f0_std, f0_range, voiced_ratio]')


## Step 10a — Collect F₀-Augmented Training Data (~20 min)

In [ ]:
# ── Step 10a: Collect F0-augmented training data ────────────────────────────
from src.pipeline import Pipeline
from src.routing.policy_learned import generate_oracle_labels
from src.lid.f0_features import extract_f0_features, is_available
from evaluation.data_loader import load_fleurs, iterate_fleurs, SUBSET_30_FLEURS
import numpy as np, pathlib

print(f'F0 extraction: {"ACTIVE" if is_available() else "DISABLED (zero vectors used)"}')

pipe = Pipeline()
pipe.load_models()

fused_probs_list, uncertainty_list, true_langs, f0_list = [], [], [], []
datasets = load_fleurs(SUBSET_30_FLEURS, split='validation', streaming=True)

n = 0
for audio, sr, fleurs_code, true_lang, ref_text in iterate_fleurs(datasets, max_per_lang=50):
    lid_out = pipe.run_lid_only(audio, sr)
    fused_probs_list.append(lid_out['fused_probs'])
    uncertainty_list.append(lid_out['uncertainty'])
    true_langs.append(true_lang)
    f0_list.append(extract_f0_features(audio, sr))
    n += 1
    if n % 100 == 0:
        print(f'  Processed {n} samples...')

pipe.unload_models()
print(f'Total samples: {n}')

# Build 11-dim base features + oracle labels
X_base, y = generate_oracle_labels(fused_probs_list, uncertainty_list, true_langs)
print(f'Base (11-dim): {X_base.shape}, labels: {np.bincount(y)}')

# Append 4 F0 dims → 15-dim total
F0_mat = np.array(f0_list, dtype=np.float32)
X_f0 = np.concatenate([X_base, F0_mat], axis=1)
print(f'F0-augmented (15-dim): {X_f0.shape}')

# Save
pathlib.Path('models').mkdir(exist_ok=True)
np.save('models/X_f0_train.npy', X_f0)
np.save('models/y_f0_train.npy', y)
np.save('models/X_f0_feats.npy', F0_mat)
print('Saved: models/X_f0_train.npy, models/y_f0_train.npy')


## Step 10b — Retrain MLP with F₀ Features (input_dim=15, ~5 min)

In [ ]:
# ── Step 10b: Retrain F0-aware MLP ──────────────────────────────────────────
from src.routing.policy_learned import LearnedRoutingPolicy
import numpy as np

X_f0 = np.load('models/X_f0_train.npy')
y    = np.load('models/y_f0_train.npy')
print(f'Training data: {X_f0.shape}, labels: {np.bincount(y)}')

policy_f0 = LearnedRoutingPolicy(
    input_dim=15,    # 6 uncertainty + 5 top probs + 4 F0
    hidden_dim=128,  # larger hidden layer for extra features
)
history_f0 = policy_f0.train_policy(X_f0, y, epochs=120, lr=0.001)
best_val = max(history_f0['val_acc'])
print(f'Best val_acc: {best_val:.4f}  |  Final val_acc: {history_f0["val_acc"][-1]:.4f}')

policy_f0.save('models/routing_policy_f0.pt')
print('Saved: models/routing_policy_f0.pt')


## Step 10c — Evaluate Phase 3 F₀-Aware Pipeline (~40 min)

In [ ]:
# ── Step 10c: Evaluate F0-aware pipeline ────────────────────────────────────
import numpy as np, json, importlib, sys
import pathlib

from src.routing.policy_learned import LearnedRoutingPolicy
from src.lid.f0_features import extract_f0_features, is_available
from src.routing.confusion_map import ConfusionMap
from src.pipeline import Pipeline
from src.routing.policy_rules import RoutingMode
from src.decoding.single_decode import decode_single
from src.decoding.multi_hypothesis import decode_multi_hypothesis
from src.decoding.fallback_decode import decode_fallback
from src.utils import PipelineOutput
from src.preprocessing import preprocess
from evaluation.evaluate import evaluate_full
from evaluation.data_loader import SUBSET_30_FLEURS


class F0Pipeline(Pipeline):
    """Pipeline subclass that injects F0 features into routing decisions."""

    def __init__(self, f0_policy_path, **kwargs):
        super().__init__(routing_policy='learned', **kwargs)
        cmap = self.routing_agent.confusion_map
        self._f0_policy = LearnedRoutingPolicy(
            input_dim=15, hidden_dim=128, confusion_map=cmap
        )
        self._f0_policy.load(f0_policy_path)
        print(f'F0 policy loaded ({f0_policy_path})')

    def run(self, audio, sr=16000, apply_vad=True):
        segments = preprocess(audio, sr, apply_vad_flag=apply_vad)
        main_audio = segments[0] if segments else audio

        # Dual LID + fusion (unchanged)
        acoustic_probs = self.acoustic_lid.predict(main_audio)
        decoder_probs  = self.decoder_lid.predict(main_audio)
        fused_probs, uncertainty = self.fusion.fuse_and_analyze(
            acoustic_probs, decoder_probs
        )

        # Extract F0 and attach to uncertainty
        f0_feats = extract_f0_features(main_audio, sr)
        uncertainty.f0_features = f0_feats  # stored on UncertaintySignals

        # Override: build 15-dim feature manually and call decide()
        # policy_learned.decide() uses self.input_dim=15 automatically
        # because it reads from uncertainty.to_vector_extended() via feature_vec
        # We monkey-patch the feature building for this one call:
        import torch
        base_vec = uncertainty.to_vector()       # 6-dim
        sprobs   = sorted(fused_probs.values(), reverse=True)[:5]
        sprobs  += [0.0] * (5 - len(sprobs))     # pad to 5
        feat15   = np.concatenate([base_vec, sprobs, f0_feats]).astype('float32')
        # Directly call the model forward instead of going through decide()
        self._f0_policy._model.eval()
        with torch.no_grad():
            import torch as _t
            logits = self._f0_policy._model(_t.from_numpy(feat15).unsqueeze(0))
            probs  = _t.softmax(logits, dim=-1)[0]
            mode_idx = probs.argmax().item()

        from src.routing.policy_rules import RoutingDecision
        mode_labels = [RoutingMode.SINGLE, RoutingMode.MULTI_HYPOTHESIS, RoutingMode.FALLBACK]
        raw_mode    = mode_labels[mode_idx]

        from src.utils import get_language_map
        lang_map = get_language_map()
        sorted_langs = [l for l in sorted(fused_probs, key=fused_probs.get, reverse=True)
                        if lang_map.asr_capable(l)]

        # Apply Phase 1+2 overrides (reuse from policy_learned)
        from src.routing.policy_rules import _apply_temperature
        cmap = self.routing_agent.confusion_map
        top1 = sorted_langs[0] if sorted_langs else 'unk'
        tau  = cmap.temperature(top1)
        tempered = _apply_temperature(fused_probs, tau)
        t_sorted = [l for l in sorted(tempered, key=tempered.get, reverse=True)
                    if lang_map.asr_capable(l)]
        t1p = tempered.get(t_sorted[0], 0.0) if t_sorted else 0.0
        t2p = tempered.get(t_sorted[1], 0.0) if len(t_sorted) > 1 else 0.0
        tgap = t1p - t2p

        if raw_mode == RoutingMode.SINGLE:
            fb_thresh = cmap.force_mode_b_threshold(top1)
            if fb_thresh > 0 and t1p < fb_thresh:
                raw_mode = RoutingMode.MULTI_HYPOTHESIS
            elif tgap < 0.10 and cmap.is_confused(top1):
                raw_mode = RoutingMode.MULTI_HYPOTHESIS

        if raw_mode == RoutingMode.SINGLE:
            candidates = t_sorted[:1]
        elif raw_mode == RoutingMode.MULTI_HYPOTHESIS:
            candidates = t_sorted[:3]
            for p in cmap.get_partners(top1):
                if p not in candidates and lang_map.asr_capable(p): candidates.append(p)
            candidates = candidates[:5]
        else:
            candidates = t_sorted[:5]

        decision = RoutingDecision(mode=raw_mode, candidate_languages=candidates,
                                   confidence=float(probs[mode_idx]),
                                   reason=f'F0-aware MLP (tau={tau:.1f})')

        all_transcripts = []
        if decision.mode == RoutingMode.SINGLE:
            lang = decision.candidate_languages[0]
            best = decode_single(main_audio, lang, self.whisper_backend,
                                 self.mms_backend, lid_confidence=decision.confidence)
            all_transcripts = [best]
        elif decision.mode == RoutingMode.MULTI_HYPOTHESIS:
            best, all_transcripts = decode_multi_hypothesis(
                main_audio, decision.candidate_languages, fused_probs,
                self.whisper_backend, self.mms_backend)
        else:
            best, all_transcripts = decode_fallback(
                main_audio, decision.candidate_languages, fused_probs,
                self.whisper_backend, self.mms_backend)

        return PipelineOutput(
            transcript=best.text, detected_language=best.language,
            confidence=best.confidence, routing_mode=decision.mode,
            candidates_considered=len(decision.candidate_languages),
            backend_used=best.backend,
            lid_distribution=dict(sorted(fused_probs.items(),
                                         key=lambda x: x[1], reverse=True)[:10]),
            all_transcripts=all_transcripts, uncertainty=uncertainty,
        )


pipe_f0 = F0Pipeline(f0_policy_path='models/routing_policy_f0.pt')
pipe_f0.load_models()

results_f0 = evaluate_full(
    pipeline=pipe_f0,
    lang_codes=SUBSET_30_FLEURS,
    split='test',
    max_per_lang=30,
    save_path='results/step10_phase3_f0.json',
)
print('\n=== Step 10c: Phase 3 F0-Aware Routing ===')
print('CER: %.4f' % results_f0.mean_cer())
print('WER: %.4f' % results_f0.mean_wer())
print('Routing:', results_f0.routing_distribution())

pipe_f0.unload_models()

with open('results/step10_phase3_f0.json') as f:
    d10 = json.load(f)
per = d10.get('per_language', {})
print('\nProblem language CER after Phase 3:')
for lang in ['urd', 'srp', 'hin', 'cmn', 'lao', 'jpn', 'yor']:
    if lang in per:
        print(f'  {lang}: CER={per[lang]["mean_cer"]:.4f}  '
              f'LID={per[lang]["lid_accuracy"]:.2f}')


---
## Step 11 — Grand Comparison: All Systems

Run any time after Steps 9b and 10c. Only reads saved JSON files — no pipeline.

In [ ]:
# ── Step 11: Grand Comparison ────────────────────────────────────────────────
import json, pathlib

def _cer_wer(path):
    try:
        with open(path) as f: d = json.load(f)
        return (d.get('overall_mean_cer', d.get('overall', float('nan'))),
                d.get('overall_mean_wer', float('nan')))
    except Exception:
        return float('nan'), float('nan')

b3_cer, _ = _cer_wer('results/B3_static_mms.json')
s8_cer, _ = _cer_wer('results/step8_track3_learned.json')

SYSTEMS = [
    ('B3 Static MMS (target)',          'results/B3_static_mms.json'),
    ('A6 Learned (no Track3)',          'results/ablations/a6_learned_policy.json'),
    ('Step8 Track3 Learned (Phase 0)',  'results/step8_track3_learned.json'),
    ('Step9a Phase1+2 Rules',           'results/step9_phase12_rules.json'),
    ('Step9b Phase1+2 Learned ★',      'results/step9_phase12_learned.json'),
    ('Step10 Phase3 F0-Aware ★★',      'results/step10_phase3_f0.json'),
]

print('=' * 72)
print(f'  {"SYSTEM":<42} {"CER":>8} {"WER":>8}  vs B3   vs S8')
print('=' * 72)
for name, path in SYSTEMS:
    cer, wer = _cer_wer(path)
    beat_b3 = '✅' if cer < b3_cer else ('❌' if cer == cer else '⏳')
    if 'target' in name:
        print(f'  {name:<42} {cer:>8.4f} {wer:>8.4f}')
        print('-' * 72)
    else:
        delta_b3 = cer - b3_cer
        delta_s8 = cer - s8_cer
        print(f'  {name:<42} {cer:>8.4f} {wer:>8.4f}  '
              f'{delta_b3:>+.4f}  {delta_s8:>+.4f}  {beat_b3}')
print('=' * 72)

# Per-language table for best available result
for best_path in ['results/step10_phase3_f0.json',
                  'results/step9_phase12_learned.json',
                  'results/step8_track3_learned.json']:
    if pathlib.Path(best_path).exists():
        break

try:
    with open(best_path) as f: d = json.load(f)
    with open('results/step8_track3_learned.json') as f: d8 = json.load(f)
    per_best = d.get('per_language', {})
    per_s8   = d8.get('per_language', {})
    if per_best:
        print(f'\nPer-language CER ({pathlib.Path(best_path).stem}) vs Step8:')
        print(f'  {"Lang":<6} {"Best":>8}  {"Step8":>8}  {"Delta":>8}  Flag')
        rows = sorted(per_best.items(), key=lambda x: x[1]['mean_cer'], reverse=True)
        for lang, info in rows:
            cv = info['mean_cer']
            s8v = per_s8.get(lang, {}).get('mean_cer', float('nan'))
            delta = cv - s8v
            flag = '🔴' if cv > 0.25 else ('🟡' if cv > 0.12 else '🟢')
            print(f'  {lang:<6} {cv:>8.4f}  {s8v:>8.4f}  {delta:>+8.4f}  {flag}')
except Exception as e:
    print(f'(Per-language breakdown: {e})')
